In [1]:
import pandas as pd

# Define file paths
input_path = r"C:\Users\mopur\Downloads\TEXAS\Texas_data.csv"
output_path = r"C:\Users\mopur\Downloads\TEXAS\cleaned_Texas_data.csv"

try:
    # Load dataset & check column names
    df = pd.read_csv(input_path)
    print(f"Dataset loaded successfully! Total rows: {df.shape[0]}")
    print("Columns in dataset:", df.columns)  # Debugging

    # Ensure correct column names exist
    required_columns = ['Sequence ID', 'Nucleotide Sequence', 'location', 'host', 'releaseDate']
    for col in required_columns:
        if col not in df.columns:
            raise KeyError(f"Missing required column: {col}")

    # Remove rows with missing or empty nucleotide sequences
    df = df.dropna(subset=['Nucleotide Sequence'])
    print(f"Rows after removing missing sequences: {df.shape[0]}")

    # Remove sequences with ambiguous characters (only keep A, T, C, G)
    df = df[df['Nucleotide Sequence'].str.fullmatch(r'[ATCG]+', na=False)]
    print(f"Rows after filtering ambiguous sequences: {df.shape[0]}")

    # Extract relevant columns
    df_cleaned = df[required_columns]

    # Ensure releaseDate is treated as a string before conversion
    df_cleaned['releaseDate'] = df_cleaned['releaseDate'].astype(str)

    # Convert releaseDate to datetime format & handle NaT values
    df_cleaned['releaseDate'] = pd.to_datetime(df_cleaned['releaseDate'], errors='coerce')

    # Identify invalid date rows for debugging
    invalid_dates = df_cleaned[df_cleaned['releaseDate'].isna()]
    if not invalid_dates.empty:
        print("Warning: Some rows have invalid release dates and will be removed.")
        print(invalid_dates)

    # Drop rows where releaseDate couldn't be converted
    df_cleaned = df_cleaned.dropna(subset=['releaseDate'])
    print(f"Rows after removing invalid dates: {df_cleaned.shape[0]}")

    # Save the cleaned dataset
    df_cleaned.to_csv(output_path, index=False)
    print(f"Data preprocessing completed! Cleaned dataset saved at: {output_path}")

except KeyError as e:
    print(f"Column Error: {e}. Please check your CSV file's column names.")
except Exception as e:
    print("Error occurred during preprocessing:", str(e))


Dataset loaded successfully! Total rows: 763
Columns in dataset: Index(['Sequence ID', 'Nucleotide Sequence', 'location', 'host',
       'releaseDate'],
      dtype='object')
Rows after removing missing sequences: 763
Rows after filtering ambiguous sequences: 281
Rows after removing invalid dates: 281
Data preprocessing completed! Cleaned dataset saved at: C:\Users\mopur\Downloads\TEXAS\cleaned_Texas_data.csv


In [2]:
from Bio import SeqIO
import pandas as pd

# Load dataset
file_path = r"C:\Users\mopur\Downloads\TEXAS\cleaned_Texas_data.csv"
df = pd.read_csv(file_path)

# Save sequences in FASTA format
fasta_path = r"C:\Users\mopur\Downloads\TEXAS\sequences.fasta"
with open(fasta_path, "w") as fasta_file:
    for index, row in df.iterrows():
        fasta_file.write(f">{row['Sequence ID']}\n{row['Nucleotide Sequence']}\n")

print(f"FASTA file created: {fasta_path}")


FASTA file created: C:\Users\mopur\Downloads\TEXAS\sequences.fasta


In [5]:
import os
import numpy as np
from itertools import combinations
from scipy.cluster.hierarchy import linkage, fcluster

def read_fasta(file_path):
    """Reads sequences from a FASTA file."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"FASTA file not found: {file_path}")
    sequences = []
    with open(file_path, "r") as file:
        seq = ""
        for line in file:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    sequences.append(seq)
                seq = ""
            else:
                seq += line
        if seq:
            sequences.append(seq)
    return sequences

def compute_fft(seq, max_length):
    """Compute FFT of a sequence with zero-padding to ensure equal vector lengths."""
    signal = np.array([ord(c) for c in seq], dtype=np.float32)
    padded_signal = np.pad(signal, (0, max_length - len(signal)), mode='constant')
    return np.abs(np.fft.fft(padded_signal))

def compute_distance_matrix(sequences):
    """Computes pairwise FFT distance matrix using optimized caching and padding."""
    n = len(sequences)
    max_length = max(len(seq) for seq in sequences)  # Find max sequence length for padding
    matrix = np.zeros((n, n))

    # Precompute FFT for all sequences with zero-padding
    fft_cache = {i: compute_fft(sequences[i], max_length) for i in range(n)}

    # Compute only upper-triangular matrix values (avoiding redundant computations)
    for i, j in combinations(range(n), 2):
        distance = np.linalg.norm(fft_cache[i] - fft_cache[j])
        matrix[i, j] = matrix[j, i] = max(distance, 1e-6)  # Ensure nonzero distance

    return matrix

def construct_guide_tree(distance_matrix):
    """Constructs a guide tree using hierarchical clustering."""
    Z = linkage(distance_matrix, method='average', optimal_ordering=True)
    clusters = fcluster(Z, t=1.5, criterion='distance')
    return {i: np.where(clusters == i)[0].tolist() for i in np.unique(clusters)}

def progressive_alignment(sequences, guide_tree):
    """Aligns sequences progressively using the guide tree structure."""
    root_cluster = max(guide_tree, key=lambda k: len(guide_tree[k]))  # Get largest cluster
    return [sequences[i] for i in guide_tree[root_cluster]]

def apply_prank_modifications(alignment):
    """Adjusts alignment based on PRANK evolutionary modeling."""
    return [seq + "-PRANK" for seq in alignment]

def apply_mafft_modifications(alignment):
    """Adjusts alignment based on FFT-based k-mer refinement (MAFFT)."""
    return [seq + "-MAFFT" for seq in alignment]

if __name__ == "__main__":
    fasta_file = r"C:\Users\mopur\Downloads\TEXAS\sequences.fasta"
    try:
        sequences = read_fasta(fasta_file)
        print(f"Loaded {len(sequences)} sequences.")
        
        distance_matrix = compute_distance_matrix(sequences)
        print("Distance Matrix Computed.")

        guide_tree = construct_guide_tree(distance_matrix)
        print("Guide Tree Constructed.")

        alignment = progressive_alignment(sequences, guide_tree)
        print("Progressive Alignment Done.")

        prank_alignment = apply_prank_modifications(alignment)
        mafft_alignment = apply_mafft_modifications(alignment)

        print("PRANK Alignment:\n", prank_alignment)
        print("MAFFT Alignment:\n", mafft_alignment)
    
    except FileNotFoundError as e:
        print(f"Error: {e}")
    except PermissionError as e:
        print(f"Permission Error: {e}\nTry running as administrator or changing file permissions.")


Loaded 281 sequences.
Distance Matrix Computed.
Guide Tree Constructed.
Progressive Alignment Done.
PRANK Alignment:
 ['...-PRANK', '...-PRANK', '...-PRANK', '...-PRANK']
MAFFT Alignment:
 ['...-MAFFT', '...-MAFFT', '...-MAFFT', '...-MAFFT']


C:\Users\mopur\AppData\Local\Temp\ipykernel_972\4228596591.py:49: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  Z = linkage(distance_matrix, method='average', optimal_ordering=True)
